# Section 4: Gathering Data

**Contents**

1. [Reading flat files](#1.-Reading-Flat-Files)
2. [Reading Excel files](#2.-Reading-Excel-Files)
3. [Connecting to a SQL database](#3.-Connecting-to-a-SQL-Database)
4. [Quickly explore a DataFrame](#4.-Quickly-Explore-a-DataFrame)

### Learning objectives and route
Read course CSV, Excel, and SQLite sources; distinguish file location from parsing options; inspect the resulting table before analysis. Follow the original demonstrations in order, treating location examples and the external database setup as instructional alternatives rather than a run-all baseline.

Optional further reading: the [study guide](../../docs/study_guide.md) and [pandas/NumPy reference](../../docs/reference/pandas_numpy.md). Optional source: lecture PDF pp. 62–76. **Predict:** which mistakes should raise an exception, and which could silently produce the wrong table?

## 1. Reading Flat Files

### a. The basics

**Run order:** The first same-folder CSV read and other path alternatives may fail. With the kernel in `notebooks/demos`, use `../../data/course_offerings.csv` for the supplied file. Skip the optional MySQL setup and continue at **4. Quickly Explore a DataFrame**; SQLite works offline.

In [1]:
import pandas as pd

**Read a file and keep the result.** `pd` is the short name created by `import pandas as pd`. `pd.read_csv(...)` reads the path inside the parentheses and returns a DataFrame (a table with labeled rows and columns). A bare call as the last line of a notebook cell displays that table, but does not give it a name for later cells. `df = pd.read_csv(...)` evaluates the call first, then binds the returned table to `df`. The first call below illustrates the same-folder path; use the supplied relative path in the next subsection to load the course file.

In [ ]:
pd.read_csv('course_offerings.csv')

In [ ]:
# Use shift-tab-tab-tab-tab (4 tabs) to view the arguments of pd.read_csv

### b. How to change the file location

### Instructional path alternatives — not baseline executable cells
Paths resolve from the kernel's working directory, not necessarily the notebook file. With the kernel rooted in `notebooks/demos`, use `../../data/course_offerings.csv`. The same-folder example, `/Users/alice/...` example, and `Data/...` example illustrate other layouts; they are not supplied locations and may fail. Run the supplied relative-path example rather than all alternatives. If even that fails, inspect the working directory before changing the parser.

In [ ]:
# If the file is in the same folder
pd.read_csv('course_offerings.csv')
# Example absolute path; replace with your own location.
pd.read_csv('/Users/alice/Desktop/Course_Materials/Data/course_offerings.csv')

In [2]:
# If the file is in another folder
pd.read_csv('../../data/course_offerings.csv')

,course_id,course_name,instructor
0,101,Intro to Python,Chris Bruehl
1,102,Intro to SQL,John Pauler
2,201,Exploratory Data Analysis,Alice Zhao
3,301,Algorithms,Chris Bruehl
4,331,Natural Language Processing,Alice Zhao


In [ ]:
# If the file is one folder below
pd.read_csv('Data/course_offerings.csv')

### c. How to change the arguments

**Predict before reading:** will `header=None` add a data row? Will `sep='|'` preserve the expected columns in a comma-separated file? A successful read is not proof of correct parsing: check column names, row count, delimiter, and inferred types. These deliberately contrasting calls do not overwrite a saved table. See [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).

**Read the arguments as instructions to the parser.** In `header=None`, `header` is an option name and `None` means “there is no header row”: pandas gives the columns integer labels and reads the original header line as data. In `sep='|'`, the quoted pipe is the field delimiter. On this comma-separated file it will not split at commas, so check whether you see one wide text column instead of the expected fields. These `name=value` pairs pass options to `read_csv`; they do not assign variables in your notebook.

Two common variations: `usecols=['course_id']` reads only the named column, and `dtype={'course_id': 'string'}` reads that identifier as text so a value like `0017` retains its leading zeros. The former controls which columns you get; the latter controls how their values are interpreted. Try both on a tiny CSV without needing a file:

```python
from io import StringIO
import pandas as pd

csv_text = StringIO("course_id,title\n0017,Python\n")
pd.read_csv(csv_text, usecols=["course_id"], dtype={"course_id": "string"})
# One column, with the text value "0017".
```

For missing data, `na_values={'fee': ['not recorded']}` tells `read_csv` to treat that text as missing in the `fee` column, in addition to the default missing-value markers. `keep_default_na=False` turns off those defaults; use it when text such as `NA` is a real code in your data, not a missing value.

In [3]:
# Tweak the header argument
pd.read_csv('../../data/course_offerings.csv', header = None)

,0,1,2
0,course_id,course_name,instructor
1,101,Intro to Python,Chris Bruehl
2,102,Intro to SQL,John Pauler
3,201,Exploratory Data Analysis,Alice Zhao
4,301,Algorithms,Chris Bruehl
5,331,Natural Language Processing,Alice Zhao


In [4]:
# Tweak the sep argument
pd.read_csv('../../data/course_offerings.csv', sep = '|')

,"course_id,course_name,instructor"
0,"101,Intro to Python,Chris Bruehl"
1,"102,Intro to SQL,John Pauler"
2,"201,Exploratory Data Analysis,Alice Zhao"
3,"301,Algorithms,Chris Bruehl"
4,"331,Natural Language Processing,Alice Zhao"


## 2. Reading Excel Files

A workbook is a container of sheets, not a single table. `sheet_name=1` selects the second sheet (zero-based), so compare its columns and grain with the default first sheet rather than assuming identical records. To select one sheet by tab name, pass that name as a string; use `sheet_name=None` to read all sheets into a dictionary of DataFrames keyed by sheet name. Reading `.xlsx` also requires the environment's Excel engine; use course setup guidance if it is missing, rather than changing data. See [read_excel](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html).

In [5]:
# The basics
pd.read_excel('../../data/Course Offerings.xlsx')

,Instructor,Title
0,Chris Bruehl,Lead Python Instructor
1,John Pauler,Chief Growth Officer
2,Alice Zhao,Data Science Instructor


In [6]:
# Tweak the sheet_name argument
pd.read_excel('../../data/Course Offerings.xlsx', sheet_name = 1)

,Course ID,Course Name,Instructor
0,101,Intro to Python,Chris Bruehl
1,102,Intro to SQL,John Pauler
2,201,Exploratory Data Analysis,Alice Zhao
3,301,Algorithms,Chris Bruehl
4,331,Natural Language Processing,Alice Zhao


## 3. Connecting to a SQL Database

**SQLite:** The supplied local `maven.db` needs no server credentials. Verify the file location first: connecting to a nonexistent SQLite path can create an empty database. **Predict:** the `WHERE` query should return a subset of the full table, with the same columns. `LIKE '1%'` matches an identifier's prefix, not a numeric range. Close connections when finished with the demonstration.

**Optional external setup:** the MySQL `!pip` and connection example is not part of the offline baseline. Its `!pip` command installs into whichever environment the shell resolves, which may differ from the notebook kernel. Configure the intended environment through course setup; do not run this installation just to follow SQLite. A driver alone does not create a server or database. `localhost`, `my_new_db`, `alice`, and `password` are dummy configuration, not working credentials. Never store real secrets in notebooks. See [read_sql](https://pandas.pydata.org/docs/reference/api/pandas.read_sql.html).

**Follow the query from file to DataFrame.** `sqlite3.connect(...)` returns a connection to the database file; `conn` stores that connection, not its rows. `pd.read_sql(query, conn)` runs the SQL string through that connection and builds a pandas DataFrame from the result. In SQL, `SELECT *` asks for every column, `FROM courses` names the table, and `WHERE` keeps only matching rows. In the next query, `%` in the `LIKE` pattern `1%` means zero or more characters after `1`, so `1`, `12`, and `105` match but `21` does not. This filter runs in SQLite, before pandas receives the results.

In [7]:
# Connect to a SQLite database

import sqlite3
conn = sqlite3.connect('../../data/maven.db')

pd.read_sql('SELECT * FROM courses', conn)

,course_id,course,instructor
0,101,Intro to Python,Chris Bruehl
1,102,Intro to SQL,John Pauler
2,201,Exploratory Data Analysis,Alice Zhao
3,301,Algorithms,Chris Bruehl
4,331,Natural Language Processing,Alice Zhao


In [8]:
# Another example of a SQL query
pd.read_sql('SELECT * FROM courses WHERE course_id LIKE "1%"', conn)

,course_id,course,instructor
0,101,Intro to Python,Chris Bruehl
1,102,Intro to SQL,John Pauler


In [ ]:
# The setup for other software like MySQL, PostgreSQL,
# Oracle and SQL Server is more involved

# 1. Install a database driver (pip install or conda install)
!pip install mysql-connector-python

# 2. Set up the database connection
import mysql.connector
conn = mysql.connector.connect(host='localhost',
                               database='my_new_db',
                               user='alice',
                               password='password')

# 3. Write SQL code in Python
pd.read_sql('SELECT * FROM courses', conn)

## 4. Quickly Explore a DataFrame

| Method | What to check |
| --- | --- |
| `head(3)` | First three rows, not a random sample |
| `shape` | Row and column counts |
| `count()` | Nonmissing values per column |
| `describe()` | Summaries of selected types; identifiers are not meaningful measurements |
| `info()` | Types and missingness; prints information and returns `None` |

### Inspect before trusting
Check that the columns correspond to course offerings and that identifiers were not accidentally interpreted as measurements. See the [glossary](../../docs/reference/glossary.md) for grain and missingness.

In [9]:
# Save the output as a DataFrame

df = pd.read_csv('../../data/course_offerings.csv')
df

,course_id,course_name,instructor
0,101,Intro to Python,Chris Bruehl
1,102,Intro to SQL,John Pauler
2,201,Exploratory Data Analysis,Alice Zhao
3,301,Algorithms,Chris Bruehl
4,331,Natural Language Processing,Alice Zhao


In [10]:
df.head(3)

,course_id,course_name,instructor
0,101,Intro to Python,Chris Bruehl
1,102,Intro to SQL,John Pauler
2,201,Exploratory Data Analysis,Alice Zhao


In [11]:
df.shape

(5, 3)

In [12]:
df.count()

course_id      5
course_name    5
instructor     5
dtype: int64

In [13]:
df.describe()

,course_id
count,5.000000
mean,207.200000
std,107.829495
min,101.000000
25%,102.000000
50%,201.000000
75%,301.000000
max,331.000000


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   course_id    5 non-null      int64 
 1   course_name  5 non-null      object
 2   instructor   5 non-null      object
dtypes: int64(1), object(2)
memory usage: 248.0+ bytes


### Recap and optional transfer
Gathering ends with a plausible schema, not merely a successful read. Record the source, parsing decisions, and one invariant (for example, the expected course identifier column).

1. Without changing `df`, compare `df.shape[0]` with `df.count()` and explain any difference.
2. Standalone parser prediction: what are the shapes below? Explain why the wrong delimiter need not raise an exception.
```python
from io import StringIO
import pandas as pd
print(pd.read_csv(StringIO('course,fee\nA,10\n')).shape)
print(pd.read_csv(StringIO('course,fee\nA,10\n'), sep='|').shape)
```